In [ ]:
import pandas as pd
import io

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

i = 1
df_24, _ = _process_data_(f'2024_data_logs/round_{i}.log')
df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

j = 2
df_23 = pd.read_csv(f"2023_data_logs/r{j}.csv", sep=';')

df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()

df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]

df_24.columns  = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]

df_test = df_24.merge(df_23, on='timestamp', how='inner')

In [ ]:
df_test.columns

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

predictor_timeframes = [1]
responder_timeframes = [1]

results = []

for i in tqdm(range(1, 5)):
    for j in range(2, 6):
        df_24, _ = _process_data_(f'2024_data_logs/round_{i}.log')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_logs/r{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        df_copy = df_test.copy()
        
        for responder_timeframe in responder_timeframes:
            for symbol in responder_symbols:
                df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
            
            for predictor_timeframe in predictor_timeframes:
                for symbol in predictor_symbols:
                    df_copy = get_future_returns(df_copy, symbol, predictor_timeframe)
                
                for predictor_symbol in predictor_symbols:
                    for responder_symbol in responder_symbols:
                        feature_col = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
                        target_col = f"{responder_symbol}_returns_in_{responder_timeframe}_its"
                        
                        df_train = df_copy[feature_col + [target_col]].dropna()
                        
                        X = df_train[feature_col]
                        y = df_train[target_col]
                        
                        model = LinearRegression(fit_intercept=False)
                        model.fit(X, y)
                        
                        y_pred = model.predict(X)
                        
                        r2 = r2_score(y, y_pred)
                        _, p_value = stats.pearsonr(y, y_pred)
                        
                        results.append({
                            '2024_day': i,
                            '2023_day': j,
                            'predictor_symbol': predictor_symbol,
                            'responder_symbol': responder_symbol,
                            'r_squared': r2,
                            'p_value': p_value,
                            'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
                        })
                
                future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
                df_copy.drop(columns=future_cols_predictor, inplace=True)
            
            future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
            df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
results_df.head(20)

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df


predictor_timeframes = [1]
responder_timeframes = [1]

results = []

for i in tqdm(range(1, 5)):
    for j in range(2, 6):
        df_24, _ = _process_data_(f'2024_data_logs/round_{i}.log')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_logs/r{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        df_copy = df_test.copy()
        
        for responder_timeframe in responder_timeframes:
            for symbol in responder_symbols:
                df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
            
            for predictor_timeframe in predictor_timeframes:
                for symbol in predictor_symbols:
                    df_copy = get_future_returns(df_copy, symbol, predictor_timeframe)
                
                for predictor_symbol in predictor_symbols:
                    for responder_symbol in responder_symbols:
                        feature_col = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its") and col.startswith(predictor_symbol)]
                        target_col = f"{responder_symbol}_returns_in_{responder_timeframe}_its"
                        
                        df_train = df_copy[feature_col + [target_col]].dropna()
                        
                        X = df_train[feature_col]
                        y = df_train[target_col]
                        
                        model = LinearRegression(fit_intercept=False)
                        model.fit(X, y)
                        
                        y_pred = model.predict(X)
                        
                        r2 = r2_score(y, y_pred)
                        _, p_value = stats.pearsonr(y, y_pred)
                        
                        results.append({
                            '2024_day': i,
                            '2023_day': j,
                            'predictor_symbol': predictor_symbol,
                            'responder_symbol': responder_symbol,
                            'r_squared': r2,
                            'p_value': p_value,
                            'equation': f"{target_col} = {model.coef_[0]:.4f} * {feature_col[0]}"
                        })
                
                future_cols_predictor = [col for col in df_copy.columns if col.endswith(f"_returns_in_{predictor_timeframe}_its")]
                df_copy.drop(columns=future_cols_predictor, inplace=True)
            
            future_cols_responder = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
            df_copy.drop(columns=future_cols_responder, inplace=True)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
import os
import json
import io
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats
from tqdm import tqdm

def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
        sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
        sandbox_log = sections[0].strip()
        activities_log = sections[1].split('Trade History:')[0]
        trade_history = json.loads(sections[1].split('Trade History:')[1])
        market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
        trade_history_df = pd.json_normalize(trade_history)
        return market_data_df, trade_history_df

results = []

for i in tqdm(range(1, 5)):
    for j in range(2, 6):
        df_24, _ = _process_data_(f'2024_data_logs/round_{i}.log')
        df_24 = df_24.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23 = pd.read_csv(f"2023_data_logs/r{j}.csv", sep=';')
        df_23 = df_23.pivot(columns='product', values='mid_price', index='timestamp').reset_index()
        
        df_23.columns = [df_23.columns[0]] + [col + '_past' for col in df_23.columns[1:]]
        df_24.columns = [df_24.columns[0]] + [col + '_curr' for col in df_24.columns[1:]]
        
        df_test = df_24.merge(df_23, on='timestamp', how='inner')
        
        predictor_symbols = [col for col in df_test.columns if col.endswith('_past')]
        responder_symbols = [col for col in df_test.columns if col.endswith('_curr')]
        
        for predictor_symbol in predictor_symbols:
            for responder_symbol in responder_symbols:
                X = df_test[[predictor_symbol]]
                y = df_test[responder_symbol]
                
                for fit_intercept in [True, False]:
                    model = LinearRegression(fit_intercept=fit_intercept)
                    model.fit(X, y)
                    
                    y_pred = model.predict(X)
                    
                    r2 = r2_score(y, y_pred)
                    _, p_value = stats.pearsonr(y, y_pred)
                    
                    if fit_intercept:
                        equation = f"{responder_symbol} = {model.intercept_:.4f} + {model.coef_[0]:.4f} * {predictor_symbol}"
                    else:
                        equation = f"{responder_symbol} = {model.coef_[0]:.4f} * {predictor_symbol}"
                    
                    results.append({
                        '2024_day': i,
                        '2023_day': j,
                        'predictor_symbol': predictor_symbol,
                        'responder_symbol': responder_symbol,
                        'fit_intercept': fit_intercept,
                        'r_squared': r2,
                        'p_value': p_value,
                        'equation': equation
                    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('r_squared', ascending=False)

print(results_df)

In [ ]:
results_df.head(20)

In [ ]:
df